# ShopBot Arena on Kaggle: all models, one notebook (Ollama + secured tunnel)

**Before running:** Settings → Accelerator: **GPU T4 x2**, Internet: **On**.

Kaggle allows one interactive GPU session at a time, so the whole arena lives here.
All five models stay loaded together on the two T4s (~28 of 30 GB), so rounds run live with no model swapping:

| Role | Model |
|---|---|
| Red strategist | Huihui-Qwen3-14B-abliterated-v2 |
| Red attacker | PIForge AgentDojo attacker 4B (converted to GGUF here) |
| Target (ShopBot's brain) | Qwen3-8B |
| Blue fast guard | Qwen3Guard-Gen-4B |
| Blue policy judge | Granite Guardian 4.1-8B |

Run Cell 1 (starts Ollama, pulls ~22 GB), Cell 2 (builds the attacker, ~10 min; re-runnable on its own), then Cell 3 (starts the API, prints the `.env` lines).
Re-running Cell 3 keeps the same tunnel URL while the session is alive.
The PIForge checkpoints have no licence on Hugging Face (the code repo is MIT). Ask the author before publishing results built on them.
Stop the session when you're done: it uses your weekly GPU hours.

In [ ]:
# Every arena model stays loaded on the two T4s at once, so rounds run live with no swapping.
#   red_strategist  Huihui-Qwen3-14B-abliterated-v2 Q4   ~9 GB   plans attacks, adapts each round
#   red_attacker    PIForge AgentDojo attacker 4B Q4_K_M ~3.5 GB writes prompt injections (RL-trained)
#   target          Qwen3-8B Q4                          ~5.5 GB ShopBot's brain, the app being attacked
#   blue_guard      Qwen3Guard-Gen-4B Q4                 ~2.6 GB fast Safe/Unsafe label on every message
#   blue_judge      Granite Guardian 4.1-8B Q4           ~5 GB   checks traffic against written rules
#   at 8K context the other four measured 26.8 GB with no room left for the attacker; 4K context + Q4 attacker => ~28.5 GB
MODELS = {
    "red_strategist": "hf.co/mradermacher/Huihui-Qwen3-14B-abliterated-v2-GGUF:Q4_K_M",
    "red_attacker":   "piforge-attacker",   # built from Hugging Face in Cell 2
    "target":         "qwen3:8b",
    "blue_guard":     "hf.co/mradermacher/Qwen3Guard-Gen-4B-GGUF:Q4_K_M",
    "blue_judge":     "hf.co/ibm-granite/granite-guardian-4.1-8b-GGUF:Q4_K_M",
}
ATTACKER_HF_REPO = "AlbertYin/agentdojo_attacker_qwen3_4b_4o_mini"   # MIT code repo: github.com/albert-y1n/PIForge

# Optional end-of-round deep review. NOT kept loaded: when called, Ollama swaps it in
# (evicting live models for a minute or two), so only use it after a round's live part.
PULL_DEEP_REVIEW = False
DEEP_REVIEW_MODEL = "gpt-oss-safeguard:20b"   # ~14 GB on disk

CONTEXT = "4096"   # per model. At 8192 the five models need ~2 GB more than two T4s have (measured 2026-10-31)

import os, sys, subprocess, time, requests

gpus = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
print(gpus or "NO GPU FOUND")
if gpus.count("GPU ") < 2:
    raise RuntimeError("Set Accelerator to 'GPU T4 x2' in the Settings panel, then restart the session.")

print("Installing zstd...")
subprocess.run("apt-get update -qq && apt-get install -y -qq zstd", shell=True)
if not os.path.exists("/usr/local/bin/ollama"):
    print("Installing Ollama...")
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
if not os.path.exists("/usr/local/bin/ollama"):
    raise RuntimeError("Ollama install failed - is Internet ON in the session panel?")

env = os.environ.copy()
env["OLLAMA_CONTEXT_LENGTH"]     = CONTEXT
env["OLLAMA_FLASH_ATTENTION"]    = "1"     # Ollama turns it off by itself where T4 can't use it
env["OLLAMA_KV_CACHE_TYPE"]      = "q8_0"
env["OLLAMA_MAX_LOADED_MODELS"]  = str(len(MODELS))   # keep every model resident together
env["OLLAMA_SCHED_SPREAD"]      = "1"     # spread each model over both T4s instead of packing one GPU
env["OLLAMA_NUM_PARALLEL"]       = "1"     # one request per model at a time keeps memory predictable
env["OLLAMA_HOST"]               = "127.0.0.1:11434"  # local only; public side goes through the auth proxy
subprocess.run("pkill -9 ollama", shell=True); time.sleep(3)
subprocess.Popen(["ollama", "serve"], env=env, start_new_session=True,
                 stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)

up = False
for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:11434/api/tags", timeout=3).status_code == 200:
            up = True; break
    except Exception: pass
    time.sleep(2)
print("Ollama up." if up else "Ollama FAILED - see /kaggle/working/ollama.log")


def pull(model):
    print(f"\nPulling {model} ...")
    p = subprocess.Popen(["ollama", "pull", model], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        sys.stdout.write("\r" + line.strip()[:90]); sys.stdout.flush()
    p.wait()
    print(f"\n{model}: " + ("ready" if p.returncode == 0 else f"PULL FAILED (code {p.returncode})"))
    return p.returncode == 0

print(subprocess.run("df -h / /tmp /kaggle/working | sort -u", shell=True, capture_output=True, text=True).stdout)
print("Needs roughly 30 GB free on / for the models (+8 GB temporary in /tmp for the attacker conversion).")

results = {}
for role, model in MODELS.items():
    if role != "red_attacker":
        results[role] = pull(model)
if PULL_DEEP_REVIEW:
    results["deep_review"] = pull(DEEP_REVIEW_MODEL)
print("\nSummary:", results)
print("Next: run Cell 2 (build the attacker), then Cell 3.")

In [ ]:
# Cell 2: build the PIForge attacker as Q4_K_M. Safe to re-run on its own: it doesn't restart Ollama.
# If Cell 3 is already running, interrupt it first (Ollama, proxy and tunnel keep running),
# run this cell, then re-run Cell 3 (the tunnel URL stays the same).
# Why Q4: at Q8 the five models need ~2 GB more than the two T4s have, and Ollama evicts the attacker.
import os, subprocess, requests
ATTACKER_HF_REPO = "AlbertYin/agentdojo_attacker_qwen3_4b_4o_mini"   # MIT code repo: github.com/albert-y1n/PIForge
NAME = "piforge-attacker"
F16 = "/tmp/attacker-f16.gguf"
Q4 = "/tmp/attacker-q4_k_m.gguf"
QUANTIZE = "/tmp/llama.cpp/build/bin/llama-quantize"

assert requests.get("http://127.0.0.1:11434/api/tags", timeout=5).status_code == 200, "Ollama is not running: run Cell 1 first"

# Replace an older build (e.g. the Q8_0 one) instead of skipping it
if NAME in subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout:
    info = subprocess.run(["ollama", "show", NAME], capture_output=True, text=True).stdout
    if "Q4_K_M" in info:
        print(f"{NAME} (Q4_K_M) already built.")
    else:
        print(f"Removing the old {NAME} build to replace it with Q4_K_M...")
        subprocess.run(["ollama", "rm", NAME])

if NAME not in subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout:
    print(subprocess.run("df -h /tmp", shell=True, capture_output=True, text=True).stdout)
    print(f"Downloading {ATTACKER_HF_REPO} (~8 GB)...")
    from huggingface_hub import snapshot_download
    src = snapshot_download(ATTACKER_HF_REPO, local_dir="/tmp/attacker_hf")   # resumes if already there

    print("Fetching llama.cpp converter and its extra dependencies...")
    subprocess.run("test -d /tmp/llama.cpp || git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp", shell=True)
    subprocess.run("pip install -q /tmp/llama.cpp/gguf-py mistral-common sentencepiece protobuf cmake", shell=True)

    if not os.path.exists(F16):
        print("Converting to GGUF (f16)...")
        conv = subprocess.run(["python", "/tmp/llama.cpp/convert_hf_to_gguf.py", src,
                               "--outfile", F16, "--outtype", "f16"], capture_output=True, text=True)
        print(conv.stdout[-1000:]); print(conv.stderr[-3000:])
        if conv.returncode != 0 or not os.path.exists(F16):
            raise RuntimeError("Conversion failed: send the error above to Claude.")

    # Ollama only quantizes safetensors imports, so quantize the GGUF with llama.cpp's own tool (CPU build, a few minutes)
    if not os.path.exists(QUANTIZE):
        print("Building llama-quantize (CPU only)...")
        b = subprocess.run("cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF "
                           "-DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /tmp/cmake.log 2>&1 && "
                           "cmake --build /tmp/llama.cpp/build --target llama-quantize -j 4 >> /tmp/cmake.log 2>&1",
                           shell=True)
        if b.returncode != 0 or not os.path.exists(QUANTIZE):
            print(open("/tmp/cmake.log").read()[-3000:])
            raise RuntimeError("Building llama-quantize failed: send the log above to Claude.")

    print("Quantizing f16 -> Q4_K_M...")
    q = subprocess.run([QUANTIZE, F16, Q4, "Q4_K_M"], capture_output=True, text=True)
    print(q.stdout[-600:]); print(q.stderr[-1500:])
    if q.returncode != 0 or not os.path.exists(Q4):
        raise RuntimeError("Quantize failed: send the error above to Claude.")

    # Qwen3 chat format, so Ollama sends messages the way the model was trained on
    open("/tmp/Modelfile", "w").write(
        f'FROM {Q4}\n'
        'TEMPLATE """{{- range .Messages }}<|im_start|>{{ .Role }}\n{{ .Content }}<|im_end|>\n{{ end }}<|im_start|>assistant\n"""\n'
        'PARAMETER stop "<|im_end|>"\n'
        'PARAMETER temperature 0.8\n')
    print("Importing into Ollama...")
    imp = subprocess.run(["ollama", "create", NAME, "-f", "/tmp/Modelfile"], capture_output=True, text=True)
    print(imp.stdout[-800:]); print(imp.stderr[-2000:])
    if imp.returncode != 0:
        raise RuntimeError("Import failed: send the error above to Claude. (Files kept in /tmp, so a retry is quick.)")
    subprocess.run(f"rm -rf /tmp/attacker_hf {F16} {Q4}", shell=True)   # only clean up once it worked
    print(f"{NAME} built.")

# Load it next to the others and keep it there
requests.post("http://127.0.0.1:11434/api/generate",
              json={"model": NAME, "keep_alive": -1, "stream": False}, timeout=900)
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)

In [ ]:
MODELS = {
    "red_strategist": "hf.co/mradermacher/Huihui-Qwen3-14B-abliterated-v2-GGUF:Q4_K_M",
    "red_attacker":   "piforge-attacker",
    "target":         "qwen3:8b",
    "blue_guard":     "hf.co/mradermacher/Qwen3Guard-Gen-4B-GGUF:Q4_K_M",
    "blue_judge":     "hf.co/ibm-granite/granite-guardian-4.1-8b-GGUF:Q4_K_M",
}   # keep the same as Cell 1
ENV_PREFIX = "ARENA"
ENV_MODELS = {"RED_STRATEGIST_MODEL": MODELS["red_strategist"], "RED_ATTACKER_MODEL": MODELS["red_attacker"],
              "TARGET_MODEL": MODELS["target"], "BLUE_GUARD_MODEL": MODELS["blue_guard"],
              "BLUE_JUDGE_MODEL": MODELS["blue_judge"]}

import os, subprocess, time, re, datetime, secrets, requests

PROXY_PORT = 8000

# 1. API key: created once, reused on reruns. Anyone without it gets 401.
key_file = "/kaggle/working/api_key.txt"
if not os.path.exists(key_file):
    open(key_file, "w").write(secrets.token_urlsafe(24))
API_KEY = open(key_file).read().strip()

# 2. Tiny auth proxy in front of Ollama (Ollama itself has no password).
proxy_src = r"""
import sys, requests
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
KEY, PORT, UP = sys.argv[1], int(sys.argv[2]), "http://127.0.0.1:11434"
SKIP = {"content-length", "transfer-encoding", "connection", "content-encoding", "host"}

class H(BaseHTTPRequestHandler):
    def _go(self):
        if self.headers.get("Authorization", "") != f"Bearer {KEY}":
            self.send_response(401); self.send_header("Content-Type", "application/json"); self.end_headers()
            self.wfile.write(b'{"error":"unauthorized"}'); return
        body = self.rfile.read(int(self.headers.get("Content-Length") or 0)) or None
        hdrs = {k: v for k, v in self.headers.items() if k.lower() not in SKIP | {"authorization"}}
        r = requests.request(self.command, UP + self.path, headers=hdrs, data=body, stream=True, timeout=900)
        self.send_response(r.status_code)
        for k, v in r.headers.items():
            if k.lower() not in SKIP: self.send_header(k, v)
        self.end_headers()
        for chunk in r.iter_content(chunk_size=None):
            if chunk: self.wfile.write(chunk); self.wfile.flush()
    do_GET = do_POST = do_DELETE = _go
    def log_message(self, *a): pass

ThreadingHTTPServer(("127.0.0.1", PORT), H).serve_forever()
"""
open("/kaggle/working/proxy.py", "w").write(proxy_src)
# No shell: `sh -c "pkill -f <path>"` matches the shell's own command line and can kill it instead
subprocess.run(["pkill", "-f", "/kaggle/working/proxy.py"]); time.sleep(1)
subprocess.Popen(["python", "/kaggle/working/proxy.py", API_KEY, str(PROXY_PORT)], start_new_session=True,
                 stdout=open("/kaggle/working/proxy.log", "w"), stderr=subprocess.STDOUT)
time.sleep(2)

# 3. cloudflared tunnel (free, no account), pointed at the PROXY
subprocess.run(
    "test -f /usr/local/bin/cloudflared || (wget -q "
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
    "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared)", shell=True)
url = None
# Reuse the last tunnel only if it really answers: the proxy behind it replies 401 without a key.
# A dead tunnel replies 530 from Cloudflare. (A pgrep check matched its own shell, so it lied.)
if os.path.exists("/kaggle/working/cf.log"):
    m = re.search(r"https://[-\w.]+trycloudflare\.com", open("/kaggle/working/cf.log").read())
    if m:
        try:
            if requests.get(m.group(0) + "/api/tags", timeout=10).status_code in (200, 401):
                url = m.group(0)
        except Exception:
            pass
if url:
    print("Reusing the running tunnel, URL unchanged.")
else:
    subprocess.run(["pkill", "-9", "-x", "cloudflared"]); time.sleep(2)
    # Detached, like the proxy, so stopping a cell can't take the tunnel down
    subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PROXY_PORT}", "--no-autoupdate"],
        stdout=open("/kaggle/working/cf.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(20):
        time.sleep(3)
        m = re.search(r"https://[-\w.]+trycloudflare\.com", open("/kaggle/working/cf.log").read())
        if m: url = m.group(0); break
    for _ in range(20):   # a fresh quick tunnel takes a few seconds to start routing
        try:
            if url and requests.get(url + "/api/tags", timeout=10).status_code in (200, 401): break
        except Exception:
            pass
        time.sleep(3)

# 4. Load + PIN every model in GPU memory (no prompt = load only; a prompt made the
#    thinking models ramble until their context filled, minutes per model)
for role, model in MODELS.items():
    print(f"Warming {role}: {model} ...")
    t = time.time()
    try:
        requests.post("http://127.0.0.1:11434/api/generate",
                      json={"model": model, "keep_alive": -1, "stream": False}, timeout=900)
        print(f"  loaded in {time.time() - t:.0f}s")
    except Exception as e:
        print("  warm-up error:", e)

# Every row should say 100% GPU. Any CPU share = lower CONTEXT in Cell 1 or drop a model.
print(subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout)

# 5. Check the proxy blocks requests without the key
try:
    no_key = requests.get(f"http://127.0.0.1:{PROXY_PORT}/api/tags", timeout=5).status_code
    with_key = requests.get(f"http://127.0.0.1:{PROXY_PORT}/api/tags",
                            headers={"Authorization": f"Bearer {API_KEY}"}, timeout=5).status_code
    print(f"Proxy check: without key -> {no_key} (want 401), with key -> {with_key} (want 200)")
except Exception as e:
    print("Proxy check failed:", e, "- see /kaggle/working/proxy.log")

print("=" * 72)
print("Paste these lines into your local .env exactly as printed (keep the key private):\n")
print(f"{ENV_PREFIX}_BASE_URL=" + ((url + "/v1") if url else "NOT_READY_check_cf.log"))
print(f"{ENV_PREFIX}_API_KEY={API_KEY}")
for env_name, model in ENV_MODELS.items():
    print(f"{env_name}={model}")
print("=" * 72)
print("Leave this cell running.\n")

# 6. Health loop
while True:
    try:
        code = requests.get("http://127.0.0.1:11434/api/tags", timeout=10).status_code
        state = "ok" if code == 200 else f"HTTP {code}"
    except Exception as e:
        state = f"DOWN ({e.__class__.__name__})"
    print(f"{datetime.datetime.now():%H:%M:%S}  ollama:{state}  {url}/v1", flush=True)
    time.sleep(120)

## Running the coordinated swarm attack

The multi-agent (swarm) campaign reuses **exactly these five models** — nothing new to serve, no cell to change here. A squad of attacker agents all share the `red_strategist` brain (each is just a different role prompt), ShopBot keeps the `target` brain, the `red_attacker` writes the planted injection, and the two guards are the blue team. The agents take **coordinated turns in sequence** (scout → plant → internal-agent hop → closer), not in parallel, so there is no extra VRAM cost and Ollama's `OLLAMA_NUM_PARALLEL=1` is fine as-is.

Once Cell 3 prints the `.env` lines and your local `.env` has them, run the campaign **from your machine** (not here):

```bash
python -m arena.swarm                 # one coordinated campaign, written to the arena DB
python -m arena.swarm --mode compare  # coordinated vs solo, prints the swarm multiplier
```

> Want the agents to act **truly in parallel** instead of taking turns? That needs `OLLAMA_NUM_PARALLEL=2`+ in Cell 1, which roughly doubles the KV-cache memory for the loaded models — over the two T4s' budget at the current 5 models / 4K context. To try it, drop one model or lower `CONTEXT` first. Sequential coordination is the default because it matches a planned, staged attack and keeps memory flat.
